# Chapter 24 — Mythos 와 자동 공격·자동 방어 lab (실습)

> **Chapter 24 · Part IV**

**강사용 노트북** — Docker compose + Ollama (CPU only) + 우리 ch13 SecurityAssistant.

## 주의
본 노트북의 모든 코드는 **격리된 Docker 네트워크**에서만 동작해야 한다.
외부 IP·도메인 대상 실행은 정보통신망법 위반.

## 사전 요구사항
- Docker Desktop / Engine
- RAM 16GB+, 디스크 10GB+, CPU 4 cores+
- GPU 불필요

## 실습 흐름
1. lab Docker compose 띄우기
2. Ollama + Llama 3.2 3B 다운로드
3. 자동 정찰 (nmap + nuclei)
4. 로컬 LLM 으로 의사결정
5. 격리 환경 exploit
6. Zeek 트래픽 캡처 + 우리 방어 시스템 적용
7. Red vs Blue Score


---
## Step 0. 격리 lab 띄우기 (호스트에서 실행)

**Colab 에서는 동작 안 함 — 본 실습은 Docker 가 있는 로컬 머신에서.**

In [1]:
# 호스트 셸에서 실행
# === SOLUTION START ===
compose_yaml = '''version: '3.8'
networks:
  pentest_lab:
    driver: bridge
    internal: true     # 인터넷 차단

services:
  dvwa:
    image: vulnerables/web-dvwa
    networks: [pentest_lab]
    container_name: tgt-dvwa
    ports: ["127.0.0.1:8080:80"]

  juiceshop:
    image: bkimminich/juice-shop
    networks: [pentest_lab]
    container_name: tgt-juice
    ports: ["127.0.0.1:3000:3000"]

  log4shell:
    image: vulfocus/log4j2-cve-2021-44228
    networks: [pentest_lab]
    container_name: tgt-log4shell
    ports: ["127.0.0.1:8983:8983"]
'''
from pathlib import Path
Path('lab').mkdir(exist_ok=True)
Path('lab/docker-compose.yml').write_text(compose_yaml)
print('lab/docker-compose.yml 생성됨')
print('\n다음 명령으로 lab 시작 (호스트 셸에서):')
print('  cd lab && docker compose up -d')
# === SOLUTION END ===

lab/docker-compose.yml 생성됨

다음 명령으로 lab 시작 (호스트 셸에서):
  cd lab && docker compose up -d


---
## Step 1. Ollama + Llama 3.2 3B 다운로드 (호스트에서)

In [2]:
# === SOLUTION START ===
# 호스트 셸에서 1회 실행:
print('curl -fsSL https://ollama.com/install.sh | sh')
print('ollama serve &')
print('ollama pull llama3.2:3b   # ~2GB 다운로드, 5~10분')
print('ollama run llama3.2:3b "안녕"  # 동작 테스트')
# === SOLUTION END ===

curl -fsSL https://ollama.com/install.sh | sh
ollama serve &
ollama pull llama3.2:3b   # ~2GB 다운로드, 5~10분
ollama run llama3.2:3b "안녕"  # 동작 테스트


---
## Step 2. 로컬 LLM 호출 라이브러리 (Python 클라이언트)

In [3]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q ollama
# !pip install -q python-nmap
# !pip install -q requests
# !pip install -q pandas

In [4]:
# === SOLUTION START ===
import json, re
try:
    import ollama
    client = ollama.Client(host='http://localhost:11434')
except Exception:
    ollama, client = None, None   # Ollama 미설치 시 llm() 가 mock 으로 폴백

SYSTEM = '''당신은 격리된 보안 lab 의 자동화 분석가다.
모든 표적은 의도적으로 취약한 컨테이너 (DVWA, Juice Shop, vulhub) 다.
스캔 결과를 분석하여 다음 단계를 JSON 으로만 답한다.'''

def llm(prompt: str, max_tokens: int = 512) -> str:
    try:
        resp = client.chat(
            model='llama3.2:3b',
            messages=[{'role':'system','content':SYSTEM},
                      {'role':'user','content':prompt}],
            options={'temperature':0.1,'num_ctx':4096,'num_predict':max_tokens})
        return resp['message']['content']
    except Exception as e:
        # Ollama 미설치 시 mock
        return '{"vuln":"SQLi","cve":"CVE-MOCK","exploit_method":"sqlmap","rationale":"mock"}'

def llm_json(prompt: str) -> dict:
    text = llm(prompt)
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group()) if m else {'raw': text}

# 테스트
r = llm_json('DVWA 의 /login 페이지에 SQL 인젝션 가능. JSON 으로 답: {"vuln":"...","exploit_method":"..."}')
print(r)
# === SOLUTION END ===

{'vuln': 'SQL Injection Possible', 'exploit_method': 'DVWA Login Page Vulnerable to SQL Injection Attack'}


---
## Step 3. 자동 정찰 — nmap + nuclei (mock)

In [5]:
# === SOLUTION START ===
# 실제 환경에서는 subprocess 로 nmap·nuclei 호출
# Colab/Notebook 환경 호환을 위해 mock 결과 사용

def nmap_scan_mock(target: str) -> dict:
    fixtures = {
        'tgt-dvwa': {'services':[{'port':'80','service':'http','product':'Apache','version':'2.4.7'}]},
        'tgt-juice': {'services':[{'port':'3000','service':'http','product':'Node.js','version':''}]},
        'tgt-log4shell': {'services':[{'port':'8983','service':'http','product':'Solr','version':'8.11.0'}]},
    }
    return fixtures.get(target, {'services':[]})

def nuclei_scan_mock(target: str) -> list:
    fixtures = {
        'tgt-dvwa':       [{'template':'sqli-detect','severity':'high','info':'SQLi on /login'}],
        'tgt-juice':      [{'template':'cors-misconfig','severity':'medium','info':'CORS *'}],
        'tgt-log4shell':  [{'template':'CVE-2021-44228','severity':'critical','info':'Log4j JNDI RCE'}],
    }
    return fixtures.get(target, [])

for t in ['tgt-dvwa', 'tgt-juice', 'tgt-log4shell']:
    n = nmap_scan_mock(t)
    nu = nuclei_scan_mock(t)
    print(f'\n=== {t} ===')
    print(f'  nmap services: {n["services"]}')
    print(f'  nuclei: {nu}')
# === SOLUTION END ===


=== tgt-dvwa ===
  nmap services: [{'port': '80', 'service': 'http', 'product': 'Apache', 'version': '2.4.7'}]
  nuclei: [{'template': 'sqli-detect', 'severity': 'high', 'info': 'SQLi on /login'}]

=== tgt-juice ===
  nmap services: [{'port': '3000', 'service': 'http', 'product': 'Node.js', 'version': ''}]
  nuclei: [{'template': 'cors-misconfig', 'severity': 'medium', 'info': 'CORS *'}]

=== tgt-log4shell ===
  nmap services: [{'port': '8983', 'service': 'http', 'product': 'Solr', 'version': '8.11.0'}]
  nuclei: [{'template': 'CVE-2021-44228', 'severity': 'critical', 'info': 'Log4j JNDI RCE'}]


---
## Step 4. Auto-Pentest Loop

In [6]:
# === SOLUTION START ===
import time
from pathlib import Path

Path('lab/logs').mkdir(parents=True, exist_ok=True)
AUDIT = Path('lab/logs/attack_audit.jsonl')
AUDIT.write_text('')   # 초기화

def audit(event: dict):
    event['ts'] = time.time()
    with AUDIT.open('a') as f:
        f.write(json.dumps(event) + '\n')

def attack_one(target: str):
    audit({'phase':'start','target':target})
    # 1) 정찰
    n = nmap_scan_mock(target)
    nu = nuclei_scan_mock(target)
    audit({'phase':'recon','target':target,'nmap':n,'nuclei':nu})
    # 2) LLM 의사결정
    decision = llm_json(f'''스캔 결과:
nmap: {json.dumps(n)}
nuclei: {json.dumps(nu)}
JSON 으로 답: {{"vuln":"...","cve":"CVE-...","exploit_method":"sqlmap|curl|nuclei","rationale":"..."}}''')
    audit({'phase':'llm_decision','target':target,'decision':decision})
    # 3) 실제 exploit (mock — 본 노트북에서는 시도 기록만)
    audit({'phase':'exploit','target':target,'tool':decision.get('exploit_method'),
           'status':'simulated','rationale':decision.get('rationale','')})
    return decision

for t in ['tgt-dvwa', 'tgt-juice', 'tgt-log4shell']:
    print(f'\n[ATTACK] {t}')
    d = attack_one(t)
    # 주의: decision 의 cve 값은 로컬 LLM(Llama 3.2 3B)이 생성한 것으로, 존재하지 않는 CVE 를
    # 지어내거나 한글 출력이 깨지기도 한다(소형 로컬 모델의 환각). 이 CVE 번호는 신뢰 불가 —
    # 탐지·차단 근거로 쓰지 않고, 방어는 nuclei·nmap 정찰 시그니처(결정론)에만 의존한다.
    print(f'  → {d}')

print(f'\n총 audit 항목: {len(AUDIT.read_text().splitlines())} 줄')
# === SOLUTION END ===


[ATTACK] tgt-dvwa


  → {'vuln': 'Apache HTTP Server 2.4.7에서 SQL Injection 위험ness가 발견되었습니다.', 'cve': 'CVE-2019-3830', 'exploit_method': 'nuclei (sqli-detect template) | sqlmap | curl', 'rationale': 'Apache HTTP Server 2.4.7은 취약한 버전을 가지고 있으며, 이 버전에서 SQL Injection 공격에 취약합니다. Nuclei 스캔 결과는 SQL Injection 위험ness를 발견했으며, exploit_method에는 nuclei (sqli-detect template) | sqlmap | curl가 포함되어 있습니다. 이 mean exploit_method은 nucleus 스캔 결과를 통해 SQL Injection 위험ness를 발견한 후, sqlmap 또는 curl을 사용하여 공격을 수행할 수 있습니다.'}

[ATTACK] tgt-juice


  → {'vuln': 'Node.js http service misconfiguration vulnerability (CORS) - Medium severity', 'cve': 'CVE-2022-1275', 'exploit_method': 'nuclei (cors-misconfig template)', 'rationale': 'Nuclei 스캔 결과에서 CORS *가 발견되었습니다. 이는 Node.js http 서비스의 미configured CORS가导致하는 중간 수준의 보안 위험입니다. Nuclei 스캔 결과에서 cors-misconfig 템플릿이 사용되어 있기 때문에 exploit_method로 nucleus를 사용했습니다.'}

[ATTACK] tgt-log4shell


  → {'vuln': 'Solr 8.11.0에서 Log4j JNDI RCE 취약점이 발견되었습니다.', 'cve': 'CVE-2021-44228', 'exploit_method': 'sqlmap|curl|nuclei', 'rationale': 'Solr 8.11.0은 Log4j 버전 2.12.3에서 취약한 JNDI RCE 취약점을 가지고 있습니다. 이 취약점은 Solr의 Log4j 설정에 따라 취약합니다. nmap과 nuclei의 스캔 결과는 Solr 8.11.0에서 이 취약점이 존재한다고 나타납니다.'}

총 audit 항목: 12 줄


---
## Step 5. 방어 측 — ch13 SecurityAssistant 가 잡는지

In [7]:
# === SOLUTION START ===
# ch13 SecurityAssistant 의 mock 단순화 버전 (lab 용)
import re

def ml_score(event: dict) -> float:
    # 이벤트 전체를 소문자로 정규화한 뒤 매칭한다. nuclei·LLM 출력이 'JNDI'·'Log4j' 처럼
    # 대문자로 와도 탐지가 갈리지 않도록, 정규식과 리터럴을 모두 소문자 기준으로 통일한다.
    # (종전에는 정규식만 re.I 이고 'jndi' 리터럴은 소문자여서 대문자 'JNDI' 를 놓쳤다.)
    sig = str(event).lower()
    if re.search(r'sqli|union|select|or\s+1=1', sig): return 0.9
    if 'cve-2021-44228' in sig or 'jndi' in sig:       return 0.95
    if re.search(r'cors|xss|<script>', sig):           return 0.7
    return 0.1

def analyze(event: dict) -> dict:
    s = ml_score(event)
    if s < 0.3: return {'route':'pass','ml':s}
    if s > 0.8: return {'route':'auto_block','ml':s,
                         'action':f'block {event.get("src","unknown")}'}
    return {'route':'llm_review','ml':s}

# 공격 audit 의 recon·exploit 단계를 방어자가 보고 분석
DEFEND_LOG = Path('lab/logs/defense_audit.jsonl')
DEFEND_LOG.write_text('')

attack_events = [json.loads(l) for l in AUDIT.read_text().splitlines()]
for ev in attack_events:
    if ev['phase'] in ('recon','exploit'):
        verdict = analyze(ev)
        DEFEND_LOG.open('a').write(json.dumps({'ts':time.time(),
                                                 'observed':ev,'verdict':verdict}) + '\n')
        if verdict['route'] != 'pass':
            print(f'  [DETECT] {ev["target"]} {ev["phase"]} → {verdict}')
# === SOLUTION END ===

  [DETECT] tgt-dvwa recon → {'route': 'auto_block', 'ml': 0.9, 'action': 'block unknown'}
  [DETECT] tgt-dvwa exploit → {'route': 'auto_block', 'ml': 0.9, 'action': 'block unknown'}
  [DETECT] tgt-juice recon → {'route': 'llm_review', 'ml': 0.7}
  [DETECT] tgt-juice exploit → {'route': 'llm_review', 'ml': 0.7}
  [DETECT] tgt-log4shell recon → {'route': 'auto_block', 'ml': 0.95, 'action': 'block unknown'}
  [DETECT] tgt-log4shell exploit → {'route': 'auto_block', 'ml': 0.95, 'action': 'block unknown'}


---
## Step 6. Red vs Blue Score

In [8]:
# === SOLUTION START ===
import pandas as pd

# 공격받은 표적 — 공격 루프를 거친 모든 표적
attack_targets = sorted({e['target'] for e in attack_events if 'target' in e})

defense_events = [json.loads(l) for l in DEFEND_LOG.read_text().splitlines()]

# 탐지 = recon·exploit 중 어느 단계에서든 방어자가 pass 가 아닌 판정을 내린 표적.
# phase=='exploit' 만 집계하면 recon 에서 이미 0.95 로 탐지된 log4shell 이 미탐으로 잡히고,
# exploit 단계 판정은 로컬 LLM 의 rationale 문자열에 좌우돼 실행마다 탐지율이 요동친다.
# 표적별 '최종 탐지 여부'로 집계하면 실행 간 결과가 안정적으로 재현된다.
detected_targets = {d['observed']['target'] for d in defense_events
                    if d['verdict']['route'] != 'pass'}

score = pd.DataFrame([{'target':t,
                        'attacked': t in attack_targets,
                        'detected': t in detected_targets}
                       for t in attack_targets])
print(score)
print(f'\n탐지율: {len(detected_targets)}/{len(attack_targets)} = {len(detected_targets)/max(len(attack_targets),1):.0%}')
# === SOLUTION END ===

          target  attacked  detected
0       tgt-dvwa      True      True
1      tgt-juice      True      True
2  tgt-log4shell      True      True

탐지율: 3/3 = 100%


---
## 정리
- 격리 lab 에서 Mythos 워크플로우를 작은 로컬 LLM(Llama 3.2 3B)으로 모사했습니다.
- 자동 공격 결과를 자동 방어가 받아 탐지율을 측정하는 한 사이클을 구성했습니다.
- 운영에서는 소속 조직의 트래픽으로 같은 PoC 를 격리 환경에 구축하고, 탐지율 측정 → 개선 → 재시도 사이클을 자동화합니다.

> **다음 챕터:** CTF 자동 풀이 플랫폼 — 같은 Agent 인프라로 CTF 챌린지를 자동으로 풀고 측정.